# Classifier comparison

Compare saved results for 10, 20, and 40 countries. Uses the original run folders or bundled results.

Set `RUN_EXPERIMENTS = True` to train a new comparison. This requires the dataset and baseline run.

In [ ]:
import csv
from html import escape
import json

from IPython.display import HTML, display
from PIL import Image

from train import PROJECT_DIR, REPO_DIR
from experiments import run_experiments

RUN_EXPERIMENTS = False

if RUN_EXPERIMENTS:
    experiment_dir = run_experiments()
else:
    completed = sorted((PROJECT_DIR / "runs" / "experiments").glob("*/completed.json"))
    experiment_dir = completed[-1].parent if completed else REPO_DIR / "results/classifier_comparison"

print("Experiment directory:", experiment_dir)

In [ ]:
with (experiment_dir / "summary.csv").open(newline="", encoding="utf-8") as stream:
    summary = list(csv.DictReader(stream))

headers = ["Countries", "Classifier", "Selected", "Validation loss", "Test accuracy +/- SD", "Macro F1", "Original 400 accuracy", "Chance"]
table_rows = []
for row in summary:
    values = [
        row["countries"], row["architecture"],
        "yes" if row["selected_by_validation"] == "True" else "",
        f"{float(row['validation_loss_mean']):.4f}",
        f"{float(row['test_accuracy_mean']):.2%} +/- {float(row['test_accuracy_sd']):.2%}",
        f"{float(row['test_macro_f1_mean']):.3f}",
        f"{float(row['shared_test_accuracy_mean']):.2%}",
        f"{float(row['chance_accuracy']):.2%}",
    ]
    table_rows.append("<tr>" + "".join("<td>" + escape(value) + "</td>" for value in values) + "</tr>")
display(HTML("<table><thead><tr>" + "".join("<th>" + name + "</th>" for name in headers) + "</tr></thead><tbody>" + "".join(table_rows) + "</tbody></table>"))
print("Normal classifiers use three training seeds on a fixed split. Shuffled-label controls use one seed.")
print("Selection used validation loss before test evaluation.")

In [ ]:
display(Image.open(experiment_dir / "comparison.png"))

## Selected checkpoint

Choose `country_count` as 10, 20, or 40. Plots show the validation-selected architecture with seed 416; the summary averages three seeds.

In [ ]:
country_count = 10
selected_dir = experiment_dir / "selected" / f"countries_{country_count}"
selection = json.loads((selected_dir / "selection.json").read_text())
print("Selected architecture:", selection["architecture"])
checkpoint_path = selected_dir / "model.pt"
if checkpoint_path.exists():
    print("Full checkpoint:", checkpoint_path)
else:
    print("This archive includes selected-model metrics and plots. Earlier checkpoints remain in the original workspace.")
display(Image.open(selected_dir / "training_curves.png"))
display(Image.open(selected_dir / "confusion_matrix.png"))

In [ ]:
print((experiment_dir / "report.txt").read_text())